# Extraccion, Transformacion y Preparacion (ETL)

El modulo `src/etl.py` carga y valida la fuente, elimina duplicados exactos, limpia espacios perifericos, codifica el objetivo y excluye campos del contacto actual que no estan disponibles al priorizar clientes antes de llamar. La imputacion, el escalado y la codificacion se ajustan luego dentro de los pipelines de scikit-learn.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "Data" / "bank-full.csv").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.etl import POST_CONTACT_FEATURES, load_bank_data, split_data
from src.modeling import make_preprocessor

df = load_bank_data()
print("Dimensiones tras limpiar duplicados:", df.shape)
print("Target codificado:", df["y"].value_counts().sort_index().to_dict())
print("Campos excluidos por ser posteriores/no disponibles al contacto:", list(POST_CONTACT_FEATURES))
splits = split_data(df, random_state=42)
print("Train:", splits.x_train.shape, "Validation:", splits.x_validation.shape, "Test:", splits.x_test.shape)
print("Proporciones de clase yes:", {
    "train": round(splits.y_train.mean(), 4),
    "validation": round(splits.y_validation.mean(), 4),
    "test": round(splits.y_test.mean(), 4),
})
print("Predictores:", splits.x_train.columns.tolist())

Dimensiones tras limpiar duplicados: (45211, 17)
Target codificado: {0: 39922, 1: 5289}
Campos excluidos por ser posteriores/no disponibles al contacto: ['duration', 'campaign', 'day', 'month', 'contact']
Train: (27126, 11) Validation: (9042, 11) Test: (9043, 11)
Proporciones de clase yes: {'train': np.float64(0.117), 'validation': np.float64(0.117), 'test': np.float64(0.117)}


In [2]:
preprocessor = make_preprocessor(splits.x_train)
x_train_ready = preprocessor.fit_transform(splits.x_train)
x_validation_ready = preprocessor.transform(splits.x_validation)
x_test_ready = preprocessor.transform(splits.x_test)
print("Matriz transformada train:", x_train_ready.shape)
print("Matriz transformada validation:", x_validation_ready.shape)
print("Matriz transformada test:", x_test_ready.shape)
print("Faltantes en la matriz original train:", int(splits.x_train.isna().sum().sum()))

Matriz transformada train: (27126, 33)
Matriz transformada validation: (9042, 33)
Matriz transformada test: (9043, 33)


## Control de fuga de informacion

Las particiones se realizan antes de cualquier ajuste de imputadores, escaladores, codificadores o selectores. El test se reserva hasta la fase de evaluacion. Para mantener una estimacion de despliegue previa al contacto, no se suministran `duration`, `campaign`, `day`, `month` ni `contact`.